# Tomography Globe with Magnets — 3D Printable Hollow Hemispheres

This notebook produces two OBJ hemisphere files suitable for full-colour 3D printing.
It uses ETOPO topography for surface displacement and a seismic tomography depth
slice for vertex colouring.

The workflow uses `model.export_hemispheres()`, which performs the full pipeline:
1. Splits the displaced outer shell at the equator (with a capped plane cut).
2. Boolean-subtracts the smooth inner sphere from each half.
3. Inserts magnet voids for snap-fit assembly.
4. Re-applies the colour recipe to the new vertices.
5. Exports two **watertight, manifold** hollow hemispheres.

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from globe3d import (
    GlobeModel,
    GeographicGrid,
    GridDisplacer,
    PolygonDisplacer,
    GridColourer,
    calculate_displacement_scale,
    generate_magnet_test_piece
)

## 1. Model Parameters

In [ ]:
# --- Globe geometry ---
outer_points = 1000000   # number of vertices in the outer shell
inner_points = 20000     # number of vertices in the inner shell
model_radius_mm = 40.0   # 80 mm diameter globe
inner_ratio = 0.8        # inner void radius as a fraction of the outer

# --- Displacement ---
vert_exagg = 50
topo_units = 'm'         # ETOPO data is in meters
tomography_displacement_scale = -1.5
displace_inner_with_tomo = True
coastline_step_mm = 0.5

# --- Boolean engine ---
boolean_engine = 'manifold'

## 2. Generate Base Spheres

In [ ]:
print("Generating GlobeModel...")
model = GlobeModel(
    n_points=outer_points,
    radius=model_radius_mm,
    hollow=True,
    inner_ratio=inner_ratio,
    inner_n_points=inner_points,
)
print(f"  Outer: {model.outer.vertices.shape[0]:,} vertices, {model.outer.faces.shape[0]:,} faces")
print(f"  Inner: {model.inner.vertices.shape[0]:,} vertices, {model.inner.faces.shape[0]:,} faces")

## 3. Load Geographic Grids

In [ ]:
dem_grid = "../inputs/ETOPO_2022_v1_60s_N90W180_surface.nc"
colour_grid = "../inputs/s40_depth_slice_2850.grd"

print("Loading ETOPO topography grid...")
topo_grid_data = GeographicGrid.from_netcdf(dem_grid, lat_var='lat', lon_var='lon', data_var='z')
print(f"  Shape: {topo_grid_data.grid.shape}")

print("Loading tomography grid...")
tomo_grid_data = GeographicGrid.from_netcdf(colour_grid, lat_var='y', lon_var='x', data_var='z')
print(f"  Shape: {tomo_grid_data.grid.shape}")

## 4. Displace Vertices

In [ ]:
coastline_shp = "../inputs/coastlines/ne_110m_land.shp"

scale = calculate_displacement_scale(model_radius_mm, vertical_exagg=vert_exagg, grid_units=topo_units)
print(f"Displacement scale factor: {scale:.6e}")

print("Displacing outer vertices with topography...")
model.outer.displace(GridDisplacer(topo_grid_data, show_progress=True), scale=scale)

print("Displacing outer vertices with tomography...")
model.outer.displace(GridDisplacer(tomo_grid_data, show_progress=True), scale=tomography_displacement_scale)

if displace_inner_with_tomo:
    print("Displacing inner vertices with tomography...")
    model.inner.displace(
        GridDisplacer(tomo_grid_data, show_progress=True),
        scale=tomography_displacement_scale * inner_ratio
    )

print("Applying step at the coastlines using shapefile...")
model.outer.displace(PolygonDisplacer(coastline_shp, displacement=coastline_step_mm))

## 5. Colour Options

In [ ]:
# Option 1: Colour according to boundaries
cmap_bounds = mcolors.ListedColormap(['red', 'white', 'blue'])
norm = mcolors.BoundaryNorm([-10, -0.5, 0.5, 10], cmap_bounds.N)
cmap = cmap_bounds
colour_kwargs = {'norm': norm}

# Option 2: cmap = 'RdBu'; colour_kwargs = {'vmin': -1, 'vmax': 1}
# Option 3: cmap = plt.get_cmap('RdBu', 7); colour_kwargs = {'vmin': -2, 'vmax': 2}

## 6. Preview Colour Map

In [ ]:
plt.figure(figsize=(10, 5))
lon_grid, lat_grid = np.meshgrid(tomo_grid_data.lons, tomo_grid_data.lats)
plt.pcolormesh(lon_grid, lat_grid, tomo_grid_data.grid, cmap=cmap, **colour_kwargs, shading='auto')
plt.colorbar(label='Value')
plt.title('2D Preview of Colour Grid')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.show()

## 7. Apply Colours & Configure Magnets

In [ ]:
# Colour outward-facing surfaces with the tomography grid
tomo_colourer = GridColourer(tomo_grid_data, colormap=cmap, **colour_kwargs)
model.outer.colour(tomo_colourer, selection='outward_facing')

# Configure magnets
model.configure_magnets(
    diameter=5.0,
    height=2.0,
    horizontal_tolerance=0.15,
    vertical_tolerance=0.10,
    vertical_offset=0.20,
    min_thickness=1.5,
    n_magnets=3,
    position=0.0,
)

## 8. Export Hemispheres

In [ ]:
model.export_hemispheres(
    '../outputs/tomo_globe_magnet_top.obj',
    '../outputs/tomo_globe_magnet_bottom.obj',
    engine=boolean_engine,
)
print("Hemispheres with magnets exported successfully!")

## 9. Visualise Final Globe in 3D

In [ ]:
from IPython.display import display

top_half, bottom_half = model.generate_hemispheres(engine=boolean_engine)

if top_half is not None and bottom_half is not None:
    print('Upper mesh: (click and drag to rotate, mouse-scroll to zoom)')
    display(top_half.show())
    print('Lower mesh: (click and drag to rotate, mouse-scroll to zoom)')
    display(bottom_half.show())

## 10. Generate Calibration Test Piece

Before printing the full globe hemispheres, print a small test piece to verify that the magnet voids have the correct fit.

In [ ]:
import os
os.makedirs('../outputs', exist_ok=True)

generate_magnet_test_piece(
    diameter=model.magnet_settings.diameter,
    height=model.magnet_settings.height,
    horizontal_tolerance=model.magnet_settings.horizontal_tolerance,
    vertical_tolerance=model.magnet_settings.vertical_tolerance,
    vertical_offset=model.magnet_settings.vertical_offset,
    min_thickness=model.magnet_settings.min_thickness,
    output_path='../outputs/magnet_test_piece.stl'
)
print("Generated calibration test piece at ../outputs/magnet_test_piece.stl")

## 11. Preview the Model in 3D

Preview the interactive 3D model:

In [ ]:
model.preview()